# Tutorial 06 — High-level orchestration: `FilterRunner` and TOML configs

So far every tutorial has built a filter by hand: *factory → `Param` → filter class → `process_filter`*. That is fine for learning. For experiments, two higher-level helpers are nicer:

- **`FilterRunner`** — one constructor that builds the model, the parameter object, and the filter, then runs it. The same code path used by the CLI and the GUI.
- **`SessionConfig` (TOML)** — serialise *filter + model + overrides* once, replay any time. Data-source flags (`--N`, `--data-filename`) stay on the CLI so a single TOML can be re-run on different trajectories.

This notebook shows both, on tiny examples.

## 1. Setup

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from prg.base_classes.filter_runner import FilterRunner, RunOptions

## 2. Minimal `FilterRunner` usage

One call replaces the model factory + `ParamLinear` + `Linear_PKF` triplet:

In [ ]:
opts = RunOptions(verbose=0, plot=False, save_history=False, base_dir=".")
runner = FilterRunner(
    filter_name="pkf",
    model_name="model_x2_y2_AQ_classic",
    mode="simulation",
    N=300,
    sKey=42,
    options=opts,
)
runner.run()

hist = runner.runner_instance.history._history
print(f"steps recorded: {len(hist)}")
print(f"first step keys: {sorted(hist[0].keys())[:6]} ...")

`runner.runner_instance` is the underlying filter (a `Linear_PKF` here). Its `history` exposes the per-step trace just as in the earlier tutorials.

## 3. `RunOptions`

The four toggles on `RunOptions`:

| field | role |
|-------|------|
| `verbose` | 0 silent, 1 INFO, 2 DEBUG |
| `plot` | display matplotlib plots after the run |
| `save_history` | dump the history as a pickle in `data/historyTracker/` |
| `base_dir` | output root (data + plots + history) |

In [ ]:
from prg.utils.metrics import compute_errors

# Same run, but with logging at INFO level.
opts2 = RunOptions(verbose=1, plot=False, save_history=False, base_dir=".")
r2 = FilterRunner(
    filter_name="epkf",
    model_name="model_x2_y1_classic",
    mode="simulation",
    N=200, sKey=0,
    options=opts2,
)
r2.run()

hist2 = r2.runner_instance.history._history
x_true = [s["xkp1"] for s in hist2 if s.get("xkp1") is not None]
x_hat  = [s["Xkp1_update"] for s in hist2 if s.get("Xkp1_update") is not None]
P_list = [s["PXXkp1_update"] for s in hist2 if s.get("PXXkp1_update") is not None]

errs = compute_errors(r2.runner_instance, x_true, x_hat, P_list)
print(f"EPKF on model_x2_y1_classic: MSE={errs['mse_total']:.4f}, NEES={errs['nees_mean']:.4f}")

## 4. Parameter sweeps via `model_kwargs`

`FilterRunner(model_kwargs={...})` rebuilds the model with the given constructor kwargs *before* the parameter object is built. This is the only way to genuinely vary scalar model parameters (`q_x`, `q_y`, `b`, etc.) — direct `setattr` after construction does **not** propagate to `param.mQ`.

Below: sweep the observation-noise variance `q_y` on `model_x1_y1_multiplicative` and watch the MSE move.

In [ ]:
def run_one(filter_name, model_name, q_y, *, N=500, seed=0):
    r = FilterRunner(
        filter_name=filter_name,
        model_name=model_name,
        mode="simulation",
        N=N, sKey=seed,
        options=RunOptions(verbose=0, plot=False, save_history=False, base_dir="."),
        model_kwargs={"q_y": q_y},
    )
    r.run()
    h = r.runner_instance.history._history
    diffs = [
        float(s["xkp1"].ravel()[0] - s["Xkp1_update"].ravel()[0]) ** 2
        for s in h if s.get("xkp1") is not None and s.get("Xkp1_update") is not None
    ]
    return float(np.mean(diffs))

q_values = np.geomspace(0.01, 2.0, 12)
mses = [run_one("epkf", "model_x1_y1_multiplicative", float(q)) for q in q_values]

plt.figure(figsize=(6, 3.2))
plt.semilogx(q_values, mses, "o-")
plt.xlabel("q_y (observation noise variance)")
plt.ylabel("MSE")
plt.title("EPKF on model_x1_y1_multiplicative — q_y sweep")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Saving and replaying with a TOML config

`SessionConfig` describes *filter + model + overrides* — everything except the data source. Save once, run any time, on any trajectory size.

In [ ]:
from pathlib import Path

from prg.gui.session import SessionConfig, save_config, load_config

cfg = SessionConfig(
    filter_name="epkf",
    model_name="model_x1_y1_multiplicative",
    overrides={"q_y": 0.5},
    label="EPKF on multiplicative, q_y=0.5",
)

cfg_path = Path("session_demo.toml")
save_config(cfg, cfg_path)
print(cfg_path.read_text())

The same TOML can now be replayed from the CLI. The data-source flags (`--N`, `--data-filename`) remain on the command line so the saved config can run against a 100-step or a 10 000-step trajectory without editing.

```bash
awesomepkf-epkf --config session_demo.toml --N 500
# or, equivalently:
python -m prg.run_nonlinear_epkf --config session_demo.toml --N 500
```

Below we invoke it from Python via `subprocess` so you can see the output inline:

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "prg.run_nonlinear_epkf",
     "--config", str(cfg_path.resolve()), "--N", "300", "--verbose", "1"],
    capture_output=True, text=True, cwd=str(Path("..").resolve()),
)
print("returncode:", result.returncode)
print("stdout (tail):")
print(result.stdout[-600:])
if result.stderr:
    print("stderr (tail):")
    print(result.stderr[-400:])

cfg_path.unlink()  # cleanup

## 6. Going further

| Task | How |
|------|-----|
| Run any of the 6 filters with one entry point | `FilterRunner(filter_name="ukf"|"upkf"|"pf"|"ppf", ...)` (add `sigmaSet=` or `n_particles=` as required) |
| Load CSV-recorded data instead of simulating | `mode="from_file"`, `data_filename="…/dataNonLinear_<model>.csv"` |
| Persist the run | `RunOptions(save_history=True)` — pickle lands in `data/historyTracker/` |
| Sweep a model scalar | `model_kwargs={"q_y": value}` — works for any constructor parameter of the model class |
| Save a session for later | `SessionConfig(...).save(Path("foo.toml"))` |
| Replay from CLI | `awesomepkf-<filter> --config foo.toml --N <…>` |

`FilterRunner` is the single recommended runner — the per-filter scripts in `prg/run_*.py` are now thin wrappers around the same dispatcher.